In [10]:
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans

# =====================================================
# Load Dataset
# =====================================================

df = pd.read_csv("../data/processed/dataset_labeled.csv")

In [11]:
print(df.isnull().sum().to_string())

bank_count_500m                  0
bus_stop_count_500m              0
cinema_count_500m                0
clinic_count_500m                0
college_count_500m               0
hospital_count_500m              0
museum_count_500m                0
office_count_500m                0
parking_space_count_500m         0
recreation_count_500m            0
retail_count_500m                0
school_count_500m                0
temple_count_500m                0
competitor_count_500m            0
avg_restaurant_rating_500m     143
avg_review_ratings_500m        143
nearest_restaurant_m             0
place_id                         0
latitude                         0
longitude                        0
search_area                      0
primary_type                  2706
searched_as                   2700
label                            0


In [12]:

# ============================================================
# STEP 1: IMPORT REQUIRED LIBRARIES
# ============================================================

# NumPy is used for mathematical calculations.
# We will use it later for the logarithm calculation in EWM.
import numpy as np

# Pandas is used to load, clean, modify, and save datasets.
import pandas as pd

# KMeans is used to divide feasibility scores into
# three data-driven groups.
from sklearn.cluster import KMeans



In [13]:

# ============================================================
# STEP 2: LOAD THE DATASET
# ============================================================

# Load the dataset created before the EWM calculation.
df = pd.read_csv(
    "../data/processed/dataset_labeled.csv"
)

# Display the number of rows and columns.
print("Dataset shape:", df.shape)

# Display the first five rows.
display(df.head())



Dataset shape: (4172, 24)


,bank_count_500m,bus_stop_count_500m,cinema_count_500m,clinic_count_500m,college_count_500m,hospital_count_500m,museum_count_500m,office_count_500m,parking_space_count_500m,recreation_count_500m,...,avg_restaurant_rating_500m,avg_review_ratings_500m,nearest_restaurant_m,place_id,latitude,longitude,search_area,primary_type,searched_as,label
0,26,1,0,44,22,9,1,54,5,21,...,4.265000,87.700000,66.357920,ChIJsZ5mY70Z6zkRRBczgL9a8ms,27.699546,85.337687,Baneshwor,restaurant,restaurant,1
1,43,4,1,46,37,13,1,66,6,16,...,4.247727,159.477273,3.883478,ChIJ5SO0EQ0Z6zkRTMnqUebrTqA,27.692262,85.336472,Baneshwor,restaurant,restaurant,1
2,57,6,3,65,49,12,1,68,9,15,...,4.231250,201.104167,35.022157,ChIJTdMmLgAZ6zkR84zXOWwdgPE,27.689064,85.334295,Baneshwor,restaurant,restaurant,1
3,5,0,0,23,6,4,0,66,4,26,...,4.371429,83.214286,5.217717,ChIJ3Ru72S8Z6zkR6JYG3tsYlGk,27.681549,85.341340,Baneshwor,restaurant,restaurant,1
4,55,6,3,66,46,11,1,65,10,15,...,4.270213,200.893617,35.022157,ChIJJbL_SlQZ6zkRFWSjunPmuVU,27.688812,85.334080,Baneshwor,restaurant,restaurant,1


In [14]:

# ============================================================
# STEP 3: DEFINE RELATED FEATURE GROUPS
# ============================================================

# Each new group-level feature is created from related
# raw location indicators.
#
# "Benefit" means:
# Higher values are considered more favorable.
#
# "Cost" means:
# Higher values are considered less favorable.

group_specs = {

    # Commercial activity around the location
    "commercial_score": {
        "columns": [
            "office_count_500m",
            "retail_count_500m",
            "bank_count_500m"
        ],
        "type": "Benefit"
    },

    # Accessibility and convenience
    "accessibility_score": {
        "columns": [
            "bus_stop_count_500m",
            "parking_space_count_500m"
        ],
        "type": "Benefit"
    },

    # Potential demand from educational institutions
    "education_score": {
        "columns": [
            "college_count_500m",
            "school_count_500m"
        ],
        "type": "Benefit"
    },

    # Activity generated by health institutions
    "health_activity_score": {
        "columns": [
            "hospital_count_500m",
            "clinic_count_500m"
        ],
        "type": "Benefit"
    },

    # Recreational and visitor attractions
    "attraction_score": {
        "columns": [
            "recreation_count_500m",
            "cinema_count_500m",
            "museum_count_500m"
        ],
        "type": "Benefit"
    },

    # Cultural activity around the location
    "cultural_activity_score": {
        "columns": [
            "temple_count_500m"
        ],
        "type": "Benefit"
    },

    # Potential market gap based on distance
    # to the nearest restaurant
    "market_gap_score": {
        "columns": [
            "nearest_restaurant_m"
        ],
        "type": "Benefit"
    },

    # Competitive pressure from nearby restaurants
    "competition_pressure_score": {
        "columns": [
            "competitor_count_500m",
            "avg_restaurant_rating_500m",
            "avg_review_ratings_500m"
        ],
        "type": "Cost"
    }
}

# Display the defined groups.
for group_name, information in group_specs.items():

    print(f"\n{group_name}")
    print("Columns:", information["columns"])
    print("Type:", information["type"])




commercial_score
Columns: ['office_count_500m', 'retail_count_500m', 'bank_count_500m']
Type: Benefit

accessibility_score
Columns: ['bus_stop_count_500m', 'parking_space_count_500m']
Type: Benefit

education_score
Columns: ['college_count_500m', 'school_count_500m']
Type: Benefit

health_activity_score
Columns: ['hospital_count_500m', 'clinic_count_500m']
Type: Benefit

attraction_score
Columns: ['recreation_count_500m', 'cinema_count_500m', 'museum_count_500m']
Type: Benefit

cultural_activity_score
Columns: ['temple_count_500m']
Type: Benefit

market_gap_score
Columns: ['nearest_restaurant_m']
Type: Benefit

competition_pressure_score
Columns: ['competitor_count_500m', 'avg_restaurant_rating_500m', 'avg_review_ratings_500m']
Type: Cost


In [15]:

# ============================================================
# STEP 4: COLLECT ALL RAW COLUMNS
# ============================================================

# Create an empty list.
raw_columns = []

# Go through every group.
for group_name, information in group_specs.items():

    # Add the raw columns from the current group.
    raw_columns.extend(
        information["columns"]
    )

# Remove duplicate column names.
raw_columns = list(
    set(raw_columns)
)

# Sort the names alphabetically.
raw_columns = sorted(
    raw_columns
)

# Display the result.
print(
    "Number of raw columns:",
    len(raw_columns)
)

print("\nRaw columns:")

for column in raw_columns:
    print("-", column)



Number of raw columns: 17

Raw columns:
- avg_restaurant_rating_500m
- avg_review_ratings_500m
- bank_count_500m
- bus_stop_count_500m
- cinema_count_500m
- clinic_count_500m
- college_count_500m
- competitor_count_500m
- hospital_count_500m
- museum_count_500m
- nearest_restaurant_m
- office_count_500m
- parking_space_count_500m
- recreation_count_500m
- retail_count_500m
- school_count_500m
- temple_count_500m


In [16]:

# ============================================================
# STEP 5: CHECK REQUIRED COLUMNS
# ============================================================

# Find columns that are required but missing
# from the dataset.
missing_columns = []

for column in raw_columns:

    if column not in df.columns:

        missing_columns.append(
            column
        )

# Display the result.
if len(missing_columns) == 0:

    print(
        "Success: All required columns "
        "are present in the dataset."
    )

else:

    print(
        "The following columns are missing:"
    )

    print(
        missing_columns
    )



Success: All required columns are present in the dataset.


In [17]:

# ============================================================
# STEP 6: CONVERT FEATURES TO NUMERIC VALUES
# ============================================================

for column in raw_columns:

    # Convert the column to numeric.
    #
    # Invalid values are converted to NaN.
    df[column] = pd.to_numeric(
        df[column],
        errors="coerce"
    )

print(
    "All selected features were converted "
    "to numeric format."
)



All selected features were converted to numeric format.


In [18]:

# ============================================================
# STEP 7: CHECK MISSING VALUES
# ============================================================

# Count missing values in every raw feature.
missing_counts = (
    df[raw_columns]
    .isnull()
    .sum()
)

print(
    "Missing values before imputation:"
)

display(
    missing_counts
)



Missing values before imputation:


avg_restaurant_rating_500m    143
avg_review_ratings_500m       143
bank_count_500m                 0
bus_stop_count_500m             0
cinema_count_500m               0
clinic_count_500m               0
college_count_500m              0
competitor_count_500m           0
hospital_count_500m             0
museum_count_500m               0
nearest_restaurant_m            0
office_count_500m               0
parking_space_count_500m        0
recreation_count_500m           0
retail_count_500m               0
school_count_500m               0
temple_count_500m               0
dtype: int64

In [19]:

# ============================================================
# STEP 8: HANDLE MISSING VALUES
# ============================================================

# Calculate the median of every raw feature.
column_medians = (
    df[raw_columns]
    .median()
)

# Replace missing values using the median
# of the corresponding column.
df[raw_columns] = (
    df[raw_columns]
    .fillna(column_medians)
)

# Check missing values again.
remaining_missing = (
    df[raw_columns]
    .isnull()
    .sum()
    .sum()
)

print(
    "Remaining missing values:",
    remaining_missing
)



Remaining missing values: 0


In [20]:

# ============================================================
# STEP 9: CREATE THE NORMALIZATION FUNCTION
# ============================================================

def normalize_feature(
    series,
    criterion_type
):

    # Find the smallest value.
    minimum_value = series.min()

    # Find the largest value.
    maximum_value = series.max()

    # Check whether every value is identical.
    #
    # This prevents division by zero.
    if maximum_value == minimum_value:

        return pd.Series(
            1.0,
            index=series.index
        )

    # Use reverse normalization for cost criteria.
    if criterion_type == "Cost":

        return (
            maximum_value - series
        ) / (
            maximum_value
            - minimum_value
        )

    # Use normal Min-Max normalization
    # for benefit criteria.
    return (
        series - minimum_value
    ) / (
        maximum_value
        - minimum_value
    )



In [22]:

# ============================================================
# STEP 10: CREATE AN EMPTY GROUP-FEATURE DATAFRAME
# ============================================================

# This table will store the new group-level features.
group_features = pd.DataFrame(
    index=df.index
)

print(
    "Empty group-feature table created."
)



Empty group-feature table created.


In [23]:

# ============================================================
# STEP 11: CREATE COMMERCIAL SCORE
# ============================================================

# Normalize office count.
office_normalized = normalize_feature(
    df["office_count_500m"],
    "Benefit"
)

# Normalize retail count.
retail_normalized = normalize_feature(
    df["retail_count_500m"],
    "Benefit"
)

# Normalize bank count.
bank_normalized = normalize_feature(
    df["bank_count_500m"],
    "Benefit"
)

# Save normalized values.
df[
    "office_count_500m_normalized"
] = office_normalized

df[
    "retail_count_500m_normalized"
] = retail_normalized

df[
    "bank_count_500m_normalized"
] = bank_normalized

# Average the three normalized indicators.
group_features[
    "commercial_score"
] = (
    office_normalized
    + retail_normalized
    + bank_normalized
) / 3

# Add the new feature to the main dataset.
df[
    "commercial_score"
] = group_features[
    "commercial_score"
]

display(
    df[
        [
            "office_count_500m",
            "retail_count_500m",
            "bank_count_500m",
            "commercial_score"
        ]
    ].head()
)



,office_count_500m,retail_count_500m,bank_count_500m,commercial_score
0,54,52,26,0.375690
1,66,80,43,0.533246
2,68,66,57,0.559305
3,66,78,5,0.395693
4,65,64,55,0.538360


In [24]:

# ============================================================
# STEP 12: CREATE ACCESSIBILITY SCORE
# ============================================================

# Normalize bus-stop count.
bus_stop_normalized = normalize_feature(
    df["bus_stop_count_500m"],
    "Benefit"
)

# Normalize parking-space count.
parking_normalized = normalize_feature(
    df["parking_space_count_500m"],
    "Benefit"
)

# Save normalized values.
df[
    "bus_stop_count_500m_normalized"
] = bus_stop_normalized

df[
    "parking_space_count_500m_normalized"
] = parking_normalized

# Average the two normalized indicators.
group_features[
    "accessibility_score"
] = (
    bus_stop_normalized
    + parking_normalized
) / 2

# Add the new feature to the main dataset.
df[
    "accessibility_score"
] = group_features[
    "accessibility_score"
]

display(
    df[
        [
            "bus_stop_count_500m",
            "parking_space_count_500m",
            "accessibility_score"
        ]
    ].head()
)



,bus_stop_count_500m,parking_space_count_500m,accessibility_score
0,1,5,0.154150
1,4,6,0.312253
2,6,9,0.468379
3,0,4,0.086957
4,6,10,0.490119


In [25]:

# ============================================================
# STEP 13: CREATE THE REMAINING GROUP FEATURES
# ============================================================

# These groups have not yet been created.
remaining_groups = {

    "education_score": {
        "columns": [
            "college_count_500m",
            "school_count_500m"
        ],
        "type": "Benefit"
    },

    "health_activity_score": {
        "columns": [
            "hospital_count_500m",
            "clinic_count_500m"
        ],
        "type": "Benefit"
    },

    "attraction_score": {
        "columns": [
            "recreation_count_500m",
            "cinema_count_500m",
            "museum_count_500m"
        ],
        "type": "Benefit"
    },

    "cultural_activity_score": {
        "columns": [
            "temple_count_500m"
        ],
        "type": "Benefit"
    },

    "market_gap_score": {
        "columns": [
            "nearest_restaurant_m"
        ],
        "type": "Benefit"
    },

    "competition_pressure_score": {
        "columns": [
            "competitor_count_500m",
            "avg_restaurant_rating_500m",
            "avg_review_ratings_500m"
        ],
        "type": "Cost"
    }
}

# Create each remaining group.
for group_name, information in (
    remaining_groups.items()
):

    normalized_columns = []

    # Normalize every source column.
    for column in information["columns"]:

        normalized_values = (
            normalize_feature(
                df[column],
                information["type"]
            )
        )

        # Save the normalized feature.
        df[
            f"{column}_normalized"
        ] = normalized_values

        # Store normalized values temporarily.
        normalized_columns.append(
            normalized_values
        )

    # Combine the normalized columns.
    normalized_table = pd.concat(
        normalized_columns,
        axis=1
    )

    # Calculate the row-wise average.
    group_features[
        group_name
    ] = normalized_table.mean(
        axis=1
    )

    # Add the group feature to the main dataset.
    df[group_name] = (
        group_features[group_name]
    )

print(
    "All remaining group features "
    "were created successfully."
)



All remaining group features were created successfully.


In [26]:

# ============================================================
# STEP 14: INSPECT THE FINAL GROUP FEATURES
# ============================================================

# List all group-level features.
criteria = list(
    group_specs.keys()
)

print(
    "Final group-level EWM criteria:"
)

for criterion in criteria:

    print("-", criterion)

# Display the first five rows.
display(
    group_features.head()
)

# Display descriptive statistics.
display(
    group_features.describe()
)



Final group-level EWM criteria:
- commercial_score
- accessibility_score
- education_score
- health_activity_score
- attraction_score
- cultural_activity_score
- market_gap_score
- competition_pressure_score


,commercial_score,accessibility_score,education_score,health_activity_score,attraction_score,cultural_activity_score,market_gap_score,competition_pressure_score
0,0.375690,0.154150,0.463542,0.434635,0.100168,0.144231,0.015377,0.654056
1,0.533246,0.312253,0.622396,0.512309,0.126142,0.125000,0.000900,0.574179
2,0.559305,0.468379,0.616146,0.621180,0.217051,0.057692,0.008116,0.555273
3,0.395693,0.086957,0.180208,0.215832,0.112554,0.163462,0.001209,0.662465
4,0.538360,0.490119,0.584375,0.611630,0.217051,0.067308,0.008116,0.554784


,commercial_score,accessibility_score,education_score,health_activity_score,attraction_score,cultural_activity_score,market_gap_score,competition_pressure_score
count,4172.000000,4172.000000,4172.000000,4172.000000,4172.000000,4172.000000,4172.000000,4172.000000
mean,0.310979,0.190440,0.236735,0.258029,0.130037,0.183538,0.036988,0.617407
std,0.192345,0.167191,0.159807,0.184269,0.121974,0.203125,0.040134,0.089622
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.320132
25%,0.154686,0.065217,0.123438,0.117148,0.048220,0.057692,0.014578,0.580750
50%,0.287036,0.136364,0.215104,0.237903,0.094276,0.115385,0.027738,0.648914
75%,0.451057,0.288538,0.319531,0.368209,0.172589,0.211538,0.047822,0.674241
max,0.882678,0.913043,1.000000,0.948005,0.628547,1.000000,1.000000,0.997290


In [27]:

# ============================================================
# STEP 15: PREPARE DATA FOR EWM
# ============================================================

# A very small number used to prevent
# division-by-zero and log(0) errors.
epsilon = 1e-12

# Calculate the sum of each group feature.
column_sums = (
    group_features
    .sum(axis=0)
)

# Replace zero totals with epsilon.
column_sums = (
    column_sums
    .replace(0, epsilon)
)

# Convert each group feature into proportions.
P = group_features.div(
    column_sums,
    axis=1
)

# Replace zero probabilities with epsilon.
P = P.replace(
    0,
    epsilon
)

# Display the probability matrix.
display(
    P.head()
)



,commercial_score,accessibility_score,education_score,health_activity_score,attraction_score,cultural_activity_score,market_gap_score,competition_pressure_score
0,0.000290,0.000194,0.000469,0.000404,0.000185,0.000188,0.000100,0.000254
1,0.000411,0.000393,0.000630,0.000476,0.000233,0.000163,0.000006,0.000223
2,0.000431,0.000590,0.000624,0.000577,0.000400,0.000075,0.000053,0.000216
3,0.000305,0.000109,0.000182,0.000200,0.000207,0.000213,0.000008,0.000257
4,0.000415,0.000617,0.000592,0.000568,0.000400,0.000088,0.000053,0.000215


In [28]:

# ============================================================
# STEP 16: CALCULATE ENTROPY
# ============================================================

# Number of locations.
number_of_locations = len(
    group_features
)

# Calculate entropy for every group feature.
entropy = -(
    P * np.log(P)
).sum(axis=0) / np.log(
    number_of_locations
)

print("Entropy values:")

display(
    entropy
)



Entropy values:


commercial_score              0.973651
accessibility_score           0.953883
education_score               0.971653
health_activity_score         0.967392
attraction_score              0.953849
cultural_activity_score       0.942178
market_gap_score              0.954186
competition_pressure_score    0.998626
dtype: float64

In [29]:

# ============================================================
# STEP 17: CALCULATE DIVERGENCE
# ============================================================

# Divergence represents the amount of
# useful variation in each criterion.
divergence = (
    1 - entropy
)

print("Divergence values:")

display(
    divergence
)



Divergence values:


commercial_score              0.026349
accessibility_score           0.046117
education_score               0.028347
health_activity_score         0.032608
attraction_score              0.046151
cultural_activity_score       0.057822
market_gap_score              0.045814
competition_pressure_score    0.001374
dtype: float64

In [30]:

# ============================================================
# STEP 18: CALCULATE EWM WEIGHTS
# ============================================================

# Convert divergence values into weights.
weights = (
    divergence
    / divergence.sum()
)

print("EWM weights:")

display(
    weights
)

print(
    "\nSum of all weights:",
    weights.sum()
)



EWM weights:


commercial_score              0.092587
accessibility_score           0.162052
education_score               0.099608
health_activity_score         0.114583
attraction_score              0.162172
cultural_activity_score       0.203184
market_gap_score              0.160987
competition_pressure_score    0.004828
dtype: float64


Sum of all weights: 1.0


In [32]:

# ============================================================
# STEP 19: CREATE AND SAVE THE WEIGHT TABLE
# ============================================================

weight_table = pd.DataFrame({

    "feature": criteria,

    "criterion_type": [
        group_specs[feature]["type"]
        for feature in criteria
    ],

    "entropy": entropy.values,

    "divergence": divergence.values,

    "weight": weights.values
})

# Sort from highest weight to lowest weight.
weight_table = (
    weight_table
    .sort_values(
        by="weight",
        ascending=False
    )
)

# Display the weight table.
display(
    weight_table
)

# Save the weight table.
weight_table.to_csv(
    "../data/processed/"
    "entropy_feasibility_weightage.csv",
    index=False
)

print(
    "EWM weight table saved successfully."
)



,feature,criterion_type,entropy,divergence,weight
5,cultural_activity_score,Benefit,0.942178,0.057822,0.203184
4,attraction_score,Benefit,0.953849,0.046151,0.162172
1,accessibility_score,Benefit,0.953883,0.046117,0.162052
6,market_gap_score,Benefit,0.954186,0.045814,0.160987
3,health_activity_score,Benefit,0.967392,0.032608,0.114583
2,education_score,Benefit,0.971653,0.028347,0.099608
0,commercial_score,Benefit,0.973651,0.026349,0.092587
7,competition_pressure_score,Cost,0.998626,0.001374,0.004828


EWM weight table saved successfully.


In [33]:

# ============================================================
# STEP 20: CALCULATE FEATURE CONTRIBUTIONS
# ============================================================

# Calculate the contribution of every
# group-level feature.

for feature in criteria:

    contribution_column = (
        f"{feature}_contribution"
    )

    df[
        contribution_column
    ] = (
        group_features[feature]
        * weights[feature]
    )

print(
    "Feature contributions calculated."
)



Feature contributions calculated.


In [34]:

# ============================================================
# STEP 21: CALCULATE FEASIBILITY SCORE
# ============================================================

# Create the list of contribution columns.
contribution_columns = []

for feature in criteria:

    contribution_columns.append(
        f"{feature}_contribution"
    )

# Add all weighted contributions.
#
# Multiply by 100 to express the result
# on an approximately 0–100 scale.
df[
    "feasibility_score"
] = (
    df[
        contribution_columns
    ]
    .sum(axis=1)
    * 100
)

# Display feasibility-score statistics.
display(
    df[
        "feasibility_score"
    ].describe()
)



count    4172.000000
mean       18.011572
std        10.583415
min         0.256604
25%         9.682061
50%        16.640595
75%        24.633487
max        49.863785
Name: feasibility_score, dtype: float64

In [35]:

# ============================================================
# STEP 22: CREATE THREE FEASIBILITY CLUSTERS
# ============================================================

# Create the KMeans model.
kmeans = KMeans(

    # Create three groups.
    n_clusters=3,

    # Make results reproducible.
    random_state=42,

    # Run KMeans with 20 initializations.
    n_init=20
)

# Assign every location to a cluster.
df["cluster"] = (
    kmeans.fit_predict(
        df[
            ["feasibility_score"]
        ]
    )
)

print(
    "KMeans clustering completed."
)

print(
    "\nCluster counts:"
)

print(
    df["cluster"]
    .value_counts()
    .sort_index()
)



KMeans clustering completed.

Cluster counts:
cluster
0     958
1    1411
2    1803
Name: count, dtype: int64


In [37]:

# ============================================================
# STEP 23: CREATE LOW, MODERATE, AND HIGH LABELS
# ============================================================

# Calculate the mean feasibility score
# for every cluster.
cluster_means = (
    df
    .groupby("cluster")
    ["feasibility_score"]
    .mean()
    .sort_values()
)

print(
    "Cluster means:"
)

display(
    cluster_means
)

# Get clusters from the lowest mean score
# to the highest mean score.
cluster_order = (
    cluster_means.index
)

# Assign meaningful labels.
label_map = {

    cluster_order[0]: "Low",

    cluster_order[1]: "Moderate",

    cluster_order[2]: "High"
}

# Add feasibility labels.
df[
    "feasibility_label"
] = (
    df["cluster"]
    .map(label_map)
)

print(
    "\nLabel distribution:"
)

print(
    df[
        "feasibility_label"
    ]
    .value_counts()
)



Cluster means:


cluster
1     6.908464
2    18.500866
0    33.444021
Name: feasibility_score, dtype: float64


Label distribution:
feasibility_label
Moderate    1803
Low         1411
High         958
Name: count, dtype: int64


In [38]:

# ============================================================
# STEP 24: DISPLAY FINAL RESULTS
# ============================================================

# Display important final columns.
final_result_columns = [

    "commercial_score",

    "accessibility_score",

    "education_score",

    "health_activity_score",

    "attraction_score",

    "cultural_activity_score",

    "market_gap_score",

    "competition_pressure_score",

    "feasibility_score",

    "cluster",

    "feasibility_label"
]

display(
    df[
        final_result_columns
    ].head()
)



,commercial_score,accessibility_score,education_score,health_activity_score,attraction_score,cultural_activity_score,market_gap_score,competition_pressure_score,feasibility_score,cluster,feasibility_label
0,0.375690,0.154150,0.463542,0.434635,0.100168,0.144231,0.015377,0.654056,20.692160,2,Moderate
1,0.533246,0.312253,0.622396,0.512309,0.126142,0.125000,0.000900,0.574179,26.944194,0,High
2,0.559305,0.468379,0.616146,0.621180,0.217051,0.057692,0.008116,0.555273,31.114502,0,High
3,0.395693,0.086957,0.180208,0.215832,0.112554,0.163462,0.001209,0.662465,14.826712,2,Moderate
4,0.538360,0.490119,0.584375,0.611630,0.217051,0.067308,0.008116,0.554784,31.042100,0,High


In [39]:

# ============================================================
# STEP 25: SAVE THE FINAL DATASET
# ============================================================

# Save the completed dataset.
df.to_csv(
    "../data/processed/"
    "dataset_labelled_with_score.csv",
    index=False
)

print(
    "Final dataset saved successfully."
)

print(
    "\nSaved files:"
)

print(
    "1. entropy_feasibility_weightage.csv"
)

print(
    "2. dataset_labelled_with_score.csv"
)



Final dataset saved successfully.

Saved files:
1. entropy_feasibility_weightage.csv
2. dataset_labelled_with_score.csv
